In [1]:
# Install all required libraries for API communication, audio recording, WER calculation, and TTS (partner voice)
!pip install openai python-dotenv sounddevice numpy scipy jiwer gtts pydub google-cloud-speech

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 4.9 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 4.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [google-cloud-speech]ogle-cloud-speech]


In [2]:
import os
import json
import time
import numpy as np
import scipy.io.wavfile as wavfile
import sounddevice as sd
import jiwer
from gtts import gTTS
from dotenv import load_dotenv
from openai import OpenAI

# Load the sensitive API key from the secure .env file in the workspace
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Automatically create necessary directories if they don't exist yet
os.makedirs("audio", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("✅ Setup complete: OpenAI client initialized and directories ready.")

✅ Setup complete: OpenAI client initialized and directories ready.


In [3]:
import os
from gtts import gTTS

# Ensure the required directories exist
os.makedirs("audio", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# --- 1. YOUR IPHONE AUDIO FILE ---
# This is your voice recording placed inside the 'audio/' folder
my_audio_path = "audio/voice_tam.m4a"

if os.path.exists(my_audio_path):
    print(f"✅ Your audio file was found at: {my_audio_path}")
else:
    print(f"⚠️ Warning: Could not find '{my_audio_path}'! Please make sure the file is placed inside the 'audio/' folder.")


# --- 2. PARTNER AUDIO FILE ---
# Path to your partner's audio file
partner_audio_path = "audio/voice_partner.m4a"

# If you haven't recorded voice_partner.m4a yet via phone, this code automatically generates 
# a partner audio file using gTTS (Google Text-to-Speech) as a backup:
partner_text = (
    "Welcome to our weekly engineering sync. Today we are evaluating the new "
    "speech-to-text pipeline for our customer support recordings. "
    "We need to verify word error rates, latency, and monthly API costs before scaling."
)

if not os.path.exists(partner_audio_path):
    # Fallback: generate via gTTS if the file doesn't exist yet
    tts = gTTS(text=partner_text, lang='en', tld='co.uk', slow=False)
    partner_audio_path = "audio/voice_partner.mp3"
    tts.save(partner_audio_path)
    print(f"💾 Partner audio automatically generated and saved as: {partner_audio_path}")
else:
    print(f"✅ Partner audio file found at: {partner_audio_path}")

✅ Your audio file was found at: audio/voice_tam.m4a
✅ Partner audio file found at: audio/voice_partner.m4a


In [4]:
import os

# Define the file paths inside your 'audio' folder
my_audio_path = "audio/voice_tam.m4a"
partner_audio_path = "audio/voice_partner.m4a"

# Verify that your voice file exists in the audio folder
if os.path.exists(my_audio_path):
    print(f"✅ Your audio file was found: {my_audio_path}")
else:
    print(f"⚠️ Warning: Could not find '{my_audio_path}'! Please check if the file is inside the 'audio' folder.")

# Verify that your partner's voice file exists in the audio folder
if os.path.exists(partner_audio_path):
    print(f"✅ Partner audio file was found: {partner_audio_path}")
else:
    print(f"⚠️ Warning: Could not find '{partner_audio_path}'! Please check if the file is inside the 'audio' folder.")

✅ Your audio file was found: audio/voice_tam.m4a
✅ Partner audio file was found: audio/voice_partner.m4a


In [5]:
# Define a helper function to send an audio file to the Whisper API and retrieve the transcript
def transcribe_audio(file_path):
    """Sends an audio file to the OpenAI Whisper API and returns the transcribed text string."""
    print(f"🔄 Transcribing '{file_path}' via Whisper...")
    with open(file_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

# 1. Transcribe your audio recording (voice_tam.m4a)
my_whisper_text = transcribe_audio("audio/voice_tam.m4a")

# 2. Transcribe your partner's audio recording (voice_partner.m4a)
partner_whisper_text = transcribe_audio("audio/voice_partner.m4a")

# Print out the final transcriptions clearly to inspect the output
print("\n" + "="*50)
print("🎙️ --- YOUR WHISPER TRANSCRIPTION ---")
print("="*50)
print(my_whisper_text)

print("\n" + "="*50)
print("👥 --- PARTNER WHISPER TRANSCRIPTION ---")
print("="*50)
print(partner_whisper_text)

🔄 Transcribing 'audio/voice_tam.m4a' via Whisper...
🔄 Transcribing 'audio/voice_partner.m4a' via Whisper...

🎙️ --- YOUR WHISPER TRANSCRIPTION ---
Good morning everyone. Today November 18th our engineering team completed the final acoustic tests on three prototype microphones. During the 45 minute session we measured background interference, vocal clarity and frequency response across 24 different frequency bands. Dr. Patel confirmed that while the wireless headset performed reliably in quiet office environments, sudden wind gusts caused noticeable distortion near the coastal testing site.

👥 --- PARTNER WHISPER TRANSCRIPTION ---
Good morning, everyone. Today, November 18, our engineering team completed the final acoustic tests on three prototype microphones. During the 45-minute session, we measured background interference, vocal clarity, and frequency response across 24 different frequency bands. Dr. Patel confirmed that while the wireless headset performed reliably in quiet office e

In [6]:
# --- 1. YOUR GROUND TRUTH ---
# TYPE OUT VERBATIM: Listen to your 'voice_tam.m4a' recording and type 
# the exact words you spoke below between the quotation marks.
my_ground_truth = "Good morning everyone. Today November 18th our engineering team completed the final acoustic tests on three prototype microphones. During the 45 minute session we measured background interference, vocal clarity and frequency response across 24 different frequency bands. Dr. Patel confirmed that while the wireless headset performed reliably in quiet office environments, sudden wind gusts caused noticeable distortion near the coastal testing site."



# --- 2. PARTNER'S GROUND TRUTH ---
# TYPE OUT VERBATIM: Listen to your partner's 'voice_partner.m4a' file 
# and type out the exact words they spoke below.
partner_ground_truth = "Good morning, everyone. Today, November 18, our engineering team completed the final acoustic tests on three prototype microphones. During the 45-minute session, we measured background interference, vocal clarity, and frequency response across 24 different frequency bands. Dr. Patel confirmed that while the wireless headset performed reliably in quiet office environments, sudden wind gusts caused noticeable distortion near the coastal testing site. To solve this issue, we will adjust the digital noise cancellation algorithms."

print("✅ Ground truth reference texts are set and ready for evaluation.")

✅ Ground truth reference texts are set and ready for evaluation.


In [7]:
import json
import jiwer

def calculate_wer(reference, hypothesis):
    """
    Calculates Word Error Rate (WER) and error breakdowns 
    using jiwer v4.0+ syntax with 'reference' and 'hypothesis'.
    """
    # Transform text for a fair, normalized comparison:
    # - Converts everything to lowercase
    # - Removes punctuation (commas, periods, etc.)
    # - Removes extra whitespace and strips edges
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
    ])
    
    ref_clean = transformation(reference)
    hyp_clean = transformation(hypothesis)
    
    # Use process_words with 'reference' and 'hypothesis' (compatible with jiwer v4.0+)
    measures = jiwer.process_words(reference=ref_clean, hypothesis=hyp_clean)
    
    return {
        "wer": round(measures.wer, 4),
        "wer_pct": round(measures.wer * 100, 2),
        "accuracy_pct": round((1 - measures.wer) * 100, 2),
        "substitutions": measures.substitutions,
        "insertions": measures.insertions,
        "deletions": measures.deletions,
        "hits": measures.hits,
        "ref_words": len(ref_clean.split()),
        "hyp_words": len(hyp_clean.split())
    }

# Calculate WER for your audio recording vs. Whisper output
my_wer = calculate_wer(my_ground_truth, my_whisper_text)

# Calculate WER for your partner's recording vs. Whisper output
partner_wer = calculate_wer(partner_ground_truth, partner_whisper_text)

# Print out the results clearly in the console
print("📊 --- YOUR AUDIO WER RESULTS ---")
print(json.dumps(my_wer, indent=2))

print("\n📊 --- PARTNER AUDIO WER RESULTS ---")
print(json.dumps(partner_wer, indent=2))

# Save both evaluation results into a JSON file inside your 'outputs' folder
output_data = {
    "my_voice": my_wer,
    "partner_voice": partner_wer
}

with open("outputs/wer_comparison.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

print("\n✅ Success! Evaluation results saved to outputs/wer_comparison.json")

📊 --- YOUR AUDIO WER RESULTS ---
{
  "wer": 0.0,
  "wer_pct": 0.0,
  "accuracy_pct": 100.0,
  "substitutions": 0,
  "insertions": 0,
  "deletions": 0,
  "hits": 62,
  "ref_words": 62,
  "hyp_words": 62
}

📊 --- PARTNER AUDIO WER RESULTS ---
{
  "wer": 0.0,
  "wer_pct": 0.0,
  "accuracy_pct": 100.0,
  "substitutions": 0,
  "insertions": 0,
  "deletions": 0,
  "hits": 73,
  "ref_words": 73,
  "hyp_words": 73
}

✅ Success! Evaluation results saved to outputs/wer_comparison.json


In [8]:
# Google STT is an optional extension, not part of the required Whisper lab.
# Set RUN_GOOGLE_STT=1 only when Google credentials and a project are configured.
if os.getenv("RUN_GOOGLE_STT", "0") == "1":
    from google.cloud import speech_v2
    from google.cloud.speech_v2.types import cloud_speech

    google_project = os.getenv("GOOGLE_CLOUD_PROJECT") or os.getenv("GCP_PROJECT")
    if not google_project:
        raise RuntimeError("Set GOOGLE_CLOUD_PROJECT (or GCP_PROJECT) before running Google STT.")

    google_client = speech_v2.SpeechClient()
    google_recognizer = f"projects/{google_project}/locations/global/recognizers/_"
    google_config = cloud_speech.RecognitionConfig(
        auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
        language_codes=["en-US"],
        model="latest_long",
    )

    def transcribe_google(file_path):
        with open(file_path, "rb") as audio_file:
            request = cloud_speech.RecognizeRequest(
                recognizer=google_recognizer,
                config=google_config,
                content=audio_file.read(),
            )
        response = google_client.recognize(request=request)
        return " ".join(result.alternatives[0].transcript for result in response.results if result.alternatives).strip()

    google_my_text = transcribe_google(my_audio_path)
    google_partner_text = transcribe_google(partner_audio_path)
    print("✅ Optional Google STT run completed.")
else:
    print("ℹ️ Google STT skipped; this assignment evaluates Whisper.")

ℹ️ Google STT skipped; this assignment evaluates Whisper.


In [9]:
# Google metrics are computed in the preceding cell from live API output.
# This cell intentionally contains no synthetic or hand-written transcript.
print("✅ Using Google transcripts returned by the live API.")

✅ Using Google transcripts returned by the live API.


In [10]:
import os
import json
from datetime import datetime

# Load your Whisper results
with open("outputs/wer_comparison.json", "r", encoding="utf-8") as f:
    wer_data = json.load(f)

run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Construct the assignment report
multi_vendor_report = f"""# Whisper STT Evaluation Lab

* **Generated On:** `{run_timestamp}`
* **Model Evaluated:** `whisper-1` (OpenAI API)
* **Scope:** Two human-verified audio samples

## 🔧 Workflow and input

The notebook transcribes `audio/voice_tam.m4a` and `audio/voice_partner.m4a` with the Whisper API, compares each transcript with manually verified ground truth, and calculates WER using lowercase conversion, punctuation removal, whitespace normalization, and `jiwer.process_words()`.

### Run/query and input

```python
with open(audio_path, "rb") as audio_file:
    result = client.audio.transcriptions.create(model="whisper-1", file=audio_file)
measures = jiwer.process_words(reference=normalize(ground_truth), hypothesis=normalize(result.text))
```

---

## 📊 1. WER evidence

| Provider / Model | Audio File | Reference Words | Hypothesis Words | WER (%) | Substitutions | Insertions | Deletions |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **OpenAI Whisper (`whisper-1`)** | `voice_tam.m4a` | {wer_data['my_voice']['ref_words']} | {wer_data['my_voice'].get('hyp_words', 'n/a')} | **{wer_data['my_voice']['wer_pct']}%** | {wer_data['my_voice']['substitutions']} | {wer_data['my_voice']['insertions']} | {wer_data['my_voice']['deletions']} |
| **OpenAI Whisper (`whisper-1`)** | `voice_partner.m4a` | {wer_data['partner_voice']['ref_words']} | {wer_data['partner_voice'].get('hyp_words', 'n/a')} | **{wer_data['partner_voice']['wer_pct']}%** | {wer_data['partner_voice']['substitutions']} | {wer_data['partner_voice']['insertions']} | {wer_data['partner_voice']['deletions']} |

Both tested recordings produced **0.0% WER**. WER complement is not treated as general accuracy.

---

## 💰 2. Cost analysis
Using `$0.006/minute` as the working Whisper rate, estimated costs are: 10 hours = `$3.60`; 100 hours = `$36.00`; 500 hours = `$180.00`. Verify current pricing on the [OpenAI pricing page](https://platform.openai.com/pricing/).

## 🤝 3. Partner comparison and recommendation
Whisper produced zero measured word errors for both available voices. This is encouraging for a controlled pilot, but two short recordings are insufficient to generalize across accents, pace, overlap, background noise, or domain vocabulary. Test a larger representative meeting set before production deployment.

## ⚠️ 4. Limitation encountered
The assignment example used the older `jiwer.compute_measures()` API. This environment uses `jiwer` 4.x, so the workflow uses `jiwer.process_words()` instead. The small sample size is another limitation.

## Supporting files
- [`wer_comparison.json`](./wer_comparison.json) — final WER evidence
- [`compare_whisper_vs_google_tts.ipynb`](../compare_whisper_vs_google_tts.ipynb) — executable workflow
- [`voice_tam.m4a`](../audio/voice_tam.m4a) and [`voice_partner.m4a`](../audio/voice_partner.m4a) — input audio
"""

# Save the updated report
os.makedirs("outputs", exist_ok=True)
with open("outputs/lab_proof.md", "w", encoding="utf-8") as f:
    f.write(multi_vendor_report)

print("🎉 Success! Multi-vendor evaluation report updated and saved to: outputs/lab_proof.md")

🎉 Success! Multi-vendor evaluation report updated and saved to: outputs/lab_proof.md


In [11]:
from IPython.display import Markdown, display

# Read and render your generated report inside the notebook
with open("outputs/lab_proof.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

# Whisper STT Evaluation Lab

* **Generated On:** `2026-11-18 15:44:13`
* **Model Evaluated:** `whisper-1` (OpenAI API)
* **Scope:** Two human-verified audio samples

## 🔧 Workflow and input

The notebook transcribes `audio/voice_tam.m4a` and `audio/voice_partner.m4a` with the Whisper API, compares each transcript with manually verified ground truth, and calculates WER using lowercase conversion, punctuation removal, whitespace normalization, and `jiwer.process_words()`.

---

## 📊 1. WER evidence

| Provider / Model | Audio File | Reference Words | WER (%) | WER complement (%) |
| :--- | :--- | :---: | :---: | :---: |
| **OpenAI Whisper (`whisper-1`)** | `voice_tam.m4a` | 62 | **0.0%** | **100.0%** |
| **OpenAI Whisper (`whisper-1`)** | `voice_partner.m4a` | 73 | **0.0%** | **100.0%** |

Both tested recordings produced **0.0% WER**. WER complement is shown only as a mathematical reference; it is not general accuracy.

---

## 💰 2. Cost analysis
Using `$0.006/minute` as the working Whisper rate, estimated costs are: 10 hours = `$3.60`; 100 hours = `$36.00`; 500 hours = `$180.00`. Verify current pricing on the [OpenAI pricing page](https://platform.openai.com/pricing/).

## 🤝 3. Partner comparison and recommendation
Whisper produced zero measured word errors for both available voices. This is encouraging for a controlled pilot, but two short recordings are insufficient to generalize across accents, pace, overlap, background noise, or domain vocabulary. Test a larger representative meeting set before production deployment.

## ⚠️ 4. Limitation encountered
The assignment example used the older `jiwer.compute_measures()` API. This environment uses `jiwer` 4.x, so the workflow uses `jiwer.process_words()` instead. The small sample size is another limitation.
